# Overview

This is the AlphaEarth counterpart of `S2_download_GEE.ipynb`. The workflow is
much shorter than the Sentinel-2 one because there is nothing to composite:

1. Load the AOI from a GeoJSON file in `AOIs/` (the same files used elsewhere in
   this project).
2. Check which years of the AlphaEarth annual collection cover that AOI.
3. Mosaic the embedding tiles for each requested year and clip to the AOI.
4. Export one 64-band GeoTIFF per year, locally for small AOIs or to Google
   Drive for anything bigger.

Everything you are likely to want to change lives in the *Configuration* cell.

# Load packages

In [1]:
import io
import os
import json
import zipfile
from pathlib import Path

import ee
import geemap
import pandas as pd
import requests

from IPython.display import display, HTML, clear_output

# Initialise GEE

You will need to authenticate your Google account the first time you run this.
`ee.Authenticate()` opens a browser window; the resulting token is cached in
`~/.config/earthengine/` so subsequent runs skip straight to `ee.Initialize()`.

In [2]:
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine authenticated and initialized")

Earth Engine already initialized


# Configuration

One cell to rule them all - change these, then run the rest of the notebook top
to bottom.

In [ ]:
# --- Area of interest ------------------------------------------------------
AOI_DIR  = 'AOIs'                # folder holding the GeoJSON AOI files
AOI_NAME = 'mimal_test'          # AOIs/<AOI_NAME>.geojson, also the file prefix

# --- AlphaEarth (Satellite Embedding V1, annual, 64 bands, 10 m) -----------
# Years to download. None -> every year the collection has over this AOI.
YEARS = [2023, 2024]
# Which of the 64 embedding bands to export. None -> all of them. A subset
# (e.g. ['A00', 'A01', 'A02']) is the easiest way to cut the file size.
AE_BANDS = None

# --- Export ----------------------------------------------------------------
EXPORT_SCALE = 10                # metres per pixel (AlphaEarth is native 10 m)
EXPORT_CRS   = 'EPSG:4326'       # lat/long, matching the rest of the pipeline
DESTINATION  = 'auto'            # 'auto' | 'local' | 'drive'
LOCAL_DIR    = 'images/RawImages'      # where local GeoTIFFs are written
DRIVE_FOLDER = 'alphaearth_images'     # Google Drive folder for big exports
LOCAL_MAX_MB = 32                # 'auto' sends anything larger to Drive
                                 # (getDownloadURL caps out around 32-48 MB)

# Import AOI

The AOI files in `AOIs/` come in two flavours - a bare GeoJSON geometry
(`mimal_test.geojson`) and a `FeatureCollection` exported from QGIS
(`north_aus_tropical_savanna*.geojson`) - so the loader handles both. For a
`FeatureCollection` the features are dissolved into a single geometry.

In [4]:
def load_aoi_from_file(json_file_path):
    """Load a GeoJSON file as a single ee.Geometry.

    Accepts a bare geometry, a Feature, or a FeatureCollection (whose features
    are dissolved into one geometry).
    """
    with open(json_file_path, 'r') as f:
        geojson = json.load(f)

    gj_type = geojson.get('type')

    if gj_type == 'FeatureCollection':
        geometries = [ee.Geometry(feat['geometry']) for feat in geojson['features']]
        if len(geometries) == 1:
            return geometries[0]
        # Dissolve the features into a single geometry.
        return ee.FeatureCollection([ee.Feature(g) for g in geometries]).geometry()

    if gj_type == 'Feature':
        return ee.Geometry(geojson['geometry'])

    # Otherwise assume it is already a geometry (Polygon, MultiPolygon, ...).
    return ee.Geometry(geojson)

In [5]:
aoi_path = os.path.join(AOI_DIR, f'{AOI_NAME}.geojson')
aoi = load_aoi_from_file(aoi_path)

print(f"AOI file:   {aoi_path}")
print(f"AOI bounds: {aoi.bounds().getInfo()['coordinates'][0]}")
print(f"AOI area:   {aoi.area(maxError=1).getInfo() / 1e6:,.0f} km2")

AOI file:   AOIs/mimal_test.geojson
AOI bounds: [[134.42770637893312, -13.572126529461158], [134.8211008161109, -13.572126529461158], [134.8211008161109, -13.264866713777478], [134.42770637893312, -13.264866713777478], [134.42770637893312, -13.572126529461158]]
AOI area:   1,449 km2


## Preview the AOI

A quick sanity check that the AOI is where you think it is. Displaying `Map`
directly works in VS Code / Jupyter; writing the widget out to HTML and
embedding it is more reliable when rendering with Quarto, which is what the cell
below does.

In [6]:
Map = geemap.Map(lite_mode=True, zoom=2)
Map.add_basemap("SATELLITE")
Map.addLayer(aoi, {'color': 'red'}, 'Area of Interest')
Map.centerObject(aoi, zoom=10)

# Clear previous outputs before displaying the new map.
clear_output(wait=True)
Map.to_html(filename='satellite_map_alphaearth_aoi.html')
HTML('satellite_map_alphaearth_aoi.html')

# AlphaEarth (Satellite Embedding V1)

[`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL)
is a 64-band, 10 m, **annual** embedding: each pixel is a unit-length vector
(band values in [-1, 1]) summarising a full year of optical, radar and elevation
data. Because it is annual there is no cloud masking or compositing to do - you
just pick the years you want.

Each annual image is stamped 1 January of the year it *covers* (the 2024 image
summarises Jan-Dec 2024), and the collection is tiled, so the tiles intersecting
the AOI are mosaicked before clipping.

In [7]:
ALPHAEARTH_ID = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'

ae_collection = ee.ImageCollection(ALPHAEARTH_ID).filterBounds(aoi)

# Which years does the collection actually cover over this AOI?
available_years = sorted({
    pd.to_datetime(t, unit='ms').year
    for t in ae_collection.aggregate_array('system:time_start').getInfo()
})
print("AlphaEarth years available for this AOI:", available_years)


def resolve_years(requested, available):
    """Validate the requested years against what the collection actually has."""
    if requested is None:
        return list(available)

    missing = [y for y in requested if y not in available]
    if missing:
        print(f"Requested years not available and skipped: {missing}")
    years = [y for y in requested if y in available]
    if not years:
        raise ValueError(f"None of {requested} are available - choose from {available}")
    return years


def alphaearth_for_year(year):
    """The AlphaEarth embedding for one year, mosaicked and clipped to the AOI."""
    image = (ae_collection
             .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
             .mosaic()
             .clip(aoi))
    if AE_BANDS is not None:
        image = image.select(AE_BANDS)
    return image.set({'year': year})


years = resolve_years(YEARS, available_years)
embeddings = {year: alphaearth_for_year(year) for year in years}

band_names = embeddings[years[0]].bandNames().getInfo()
print(f"Downloading {len(years)} year(s): {years}")
print(f"{len(band_names)} bands: {band_names[:5]} ... {band_names[-2:]}")

AlphaEarth years available for this AOI: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
64 bands: ['A00', 'A01', 'A02', 'A03', 'A04'] ... ['A62', 'A63']


## Visual check

There is no meaningful "true colour" for a 64-dimensional embedding - mapping
three arbitrary axes to RGB just makes structurally similar areas look similar,
which is still a useful check that the imagery covers the AOI and that different
years differ where you would expect them to.

In [8]:
viz_bands = [band_names[i] for i in (1, 16, 9)]
viz = {'bands': viz_bands, 'min': -0.3, 'max': 0.3}

Map2 = geemap.Map(lite_mode=True, zoom=2)
Map2.add_basemap('SATELLITE')

for year in years:
    Map2.addLayer(embeddings[year], viz,
                  f'AlphaEarth {year} ({"/".join(viz_bands)})',
                  shown=(year == years[-1]))

Map2.addLayer(aoi, {'color': 'red'}, 'AOI')
Map2.centerObject(aoi, zoom=10)

clear_output(wait=True)
Map2.to_html(filename='satellite_map_alphaearth.html')
HTML('satellite_map_alphaearth.html')

# Exporting

Two routes out of Earth Engine:

- **Local** (`image.getDownloadURL`) - synchronous, the file lands in
  `LOCAL_DIR` immediately, but Earth Engine caps a single request at roughly
  32-48 MB, so this only works for small rasters (small AOI, few bands, or a
  coarse `EXPORT_SCALE`).
- **Drive** (`ee.batch.Export.image.toDrive`) - asynchronous batch task, no
  practical size limit. Very large rasters are automatically split into multiple
  numbered GeoTIFFs.

`DESTINATION = 'auto'` estimates the output size and picks for you. Be warned:
64 float32 bands at 10 m is ~256 MB per 100 km2, so anything beyond a small AOI
has to go to Drive - and coarsening `EXPORT_SCALE` or setting `AE_BANDS` to a
subset is worth considering if you don't need the full resolution or all 64
dimensions.

The embeddings are left as float32: the values are already unit-scaled and the
whole point of them is fine-grained numerical structure, so packing them into
int16 is a false economy.

In [9]:
# Started Drive tasks accumulate here so they can be monitored later.
TASKS = []


def estimate_size_mb(image, region, scale, bytes_per_pixel=4):
    """Rough uncompressed size of an export, in MB."""
    n_bands = image.bandNames().size().getInfo()
    n_pixels = region.area(maxError=1).getInfo() / (scale ** 2)
    return n_pixels * n_bands * bytes_per_pixel / 1e6


def export_image(image, name, region=None, scale=None, destination=None):
    """Export an image locally or to Drive, choosing automatically if asked."""
    region = aoi if region is None else region
    scale = EXPORT_SCALE if scale is None else scale
    destination = (DESTINATION if destination is None else destination).lower()

    size_mb = estimate_size_mb(image, region, scale)
    print(f"\n{name}: ~{size_mb:,.0f} MB uncompressed at {scale} m (float32)")

    if destination == 'auto':
        destination = 'local' if size_mb <= LOCAL_MAX_MB else 'drive'
        print(f"  -> auto-selected '{destination}' (local limit {LOCAL_MAX_MB} MB)")

    if destination == 'local':
        out_dir = Path(LOCAL_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{name}.tif"

        url = image.getDownloadURL({
            'scale': scale,
            'crs': EXPORT_CRS,
            'region': region,
            'format': 'GEO_TIFF',
            'filePerBand': False,      # one multi-band GeoTIFF, not one per band
        })
        response = requests.get(url, timeout=600)
        response.raise_for_status()

        # Earth Engine sometimes hands back a zipped GeoTIFF rather than a bare
        # one ('PK' is the zip magic number), so handle both.
        if response.content[:2] == b'PK':
            with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
                tif_name = next(n for n in zf.namelist() if n.lower().endswith('.tif'))
                out_path.write_bytes(zf.read(tif_name))
        else:
            out_path.write_bytes(response.content)

        print(f"  -> saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
        return out_path

    if destination == 'drive':
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=name[:100],        # EE caps description length
            folder=DRIVE_FOLDER,
            fileNamePrefix=name,
            region=region,
            scale=scale,
            crs=EXPORT_CRS,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True},
        )
        task.start()
        TASKS.append(task)
        print(f"  -> started Drive export to '{DRIVE_FOLDER}/{name}' (task {task.id})")
        return task

    raise ValueError("destination must be 'auto', 'local' or 'drive'")

# Run the exports

One GeoTIFF per year. File names carry the AOI name and the year so that
different runs and different AOIs don't overwrite each other.

In [10]:
band_tag = 'all' if AE_BANDS is None else f'{len(band_names)}band'

exports = {}
for year in years:
    exports[year] = export_image(
        embeddings[year],
        name=f"{AOI_NAME}_alphaearth_{year}_{band_tag}",
    )


mimal_test_alphaearth_2020_all: ~3,710 MB uncompressed at 10 m (float32)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'alphaearth_images/mimal_test_alphaearth_2020_all' (task ZDABD5RVLIFCHMTBAJPJ75AX)

mimal_test_alphaearth_2021_all: ~3,710 MB uncompressed at 10 m (float32)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'alphaearth_images/mimal_test_alphaearth_2021_all' (task O3PZXA5LDHIO7O3V6D2Z7MV6)

mimal_test_alphaearth_2022_all: ~3,710 MB uncompressed at 10 m (float32)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'alphaearth_images/mimal_test_alphaearth_2022_all' (task NUY73HBTKAZJZPDQLV7UWDHN)

mimal_test_alphaearth_2023_all: ~3,710 MB uncompressed at 10 m (float32)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'alphaearth_images/mimal_test_alphaearth_2023_all' (task UCLHYFXGM6WRLIJEBZAZ5IYI)

mimal_test_alphaearth_2024_all: ~3,710 MB uncompressed at 10 m (flo

## Monitor the Drive tasks

Drive exports run asynchronously on Google's servers - this notebook can be
closed and the tasks will keep going. Re-run the cell below to check progress,
or watch them at [code.earthengine.google.com](https://code.earthengine.google.com/tasks).
Finished files appear in Google Drive under `DRIVE_FOLDER`.

In [14]:
def print_task_status(tasks=None):
    """Print the current state of each export task started in this session."""
    tasks = TASKS if tasks is None else tasks
    if not tasks:
        print("No Drive tasks started in this session.")
        return
    for task in tasks:
        status = task.status()
        line = f"{status['description']:<55} {status['state']:<12}"
        if status.get('error_message'):
            line += f" {status['error_message']}"
        print(line)


print_task_status()

mimal_test_alphaearth_2020_all                          CANCEL_REQUESTED
mimal_test_alphaearth_2021_all                          CANCEL_REQUESTED
mimal_test_alphaearth_2022_all                          CANCEL_REQUESTED
mimal_test_alphaearth_2023_all                          RUNNING     
mimal_test_alphaearth_2024_all                          READY       
